# TabPFN fine-tuning (Google Colab, T4 GPU)

Fine-tunes the pretrained TabPFN v3.5 on our data and compares it with the plain TabPFN on the shared folds. Fine-tuning runs inside each fold (its 4 training folds only, 10% of them held out for early stopping), so the comparison is honest.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Secrets panel (key icon on the left) → add `TABPFN_TOKEN` with the Prior Labs API key, and allow this notebook to access it. Never paste the key into a cell.
3. The upload cell opens a file picker until all 4 files are there: first `X_train_log_knn.csv`, `X_test_log_knn.csv` and `y_train.csv` (project folder), then `folds.csv` (`Stacking/`).

**After running:** put `oof_tabpfn_ft.csv` in `Stacking/` and `submission_tabpfn_ft.csv` in `TabPFN/`, and save this notebook (with its outputs) in `TabPFN/`.

In [1]:
!pip install -q tabpfn==9.2.0

In [2]:
import os
import torch
from google.colab import userdata

os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")   # from Colab's Secrets panel, never written in the notebook
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

TimeoutException: Requesting secret TABPFN_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
# Upload the 4 data files from the repo. The file picker opens again until all 4 are here:
#   1st time: X_train_log_knn.csv, X_test_log_knn.csv, y_train.csv  (project folder)
#   2nd time: folds.csv                                             (Stacking/)
from google.colab import files

needed = {"X_train_log_knn.csv", "X_test_log_knn.csv", "y_train.csv", "folds.csv"}
while missing := needed - set(os.listdir(".")):
    print("still missing:", sorted(missing))
    files.upload()
print("all 4 files are here")

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from tabpfn import TabPFNRegressor
from tabpfn.finetuning import FinetunedTabPFNRegressor

X = pd.read_csv("X_train_log_knn.csv", index_col="id")
X_test = pd.read_csv("X_test_log_knn.csv", index_col="id")
y = pd.read_csv("y_train.csv", index_col="id").loc[X.index, "y"]
fold = pd.read_csv("folds.csv", index_col="id").loc[X.index, "fold"]
print(f"train {X.shape} | test {X_test.shape} | y {y.shape}")

In [ ]:
# Plain vs fine-tuned TabPFN on the shared folds (both on the same GPU, so the difference comes only from fine-tuning)

oof_plain = pd.Series(np.nan, index=X.index, name="tabpfn")
oof_ft = pd.Series(np.nan, index=X.index, name="tabpfn_ft")
for f in range(5):
    train, val = fold != f, fold == f
    start = time.time()
    oof_plain[val] = TabPFNRegressor(device="cuda", random_state=0).fit(X[train], y[train]).predict(X[val])
    ft = FinetunedTabPFNRegressor(device="cuda", random_state=0).fit(X[train], y[train])
    oof_ft[val] = ft.predict(X[val])
    print(f"fold {f}: plain {r2_score(y[val], oof_plain[val]):.4f} | fine-tuned {r2_score(y[val], oof_ft[val]):.4f} "
          f"({time.time() - start:.0f} s)")

diff = np.array([r2_score(y[fold == f], oof_ft[fold == f]) - r2_score(y[fold == f], oof_plain[fold == f]) for f in range(5)])
print(f"out-of-fold R2: plain {r2_score(y, oof_plain):.4f} | fine-tuned {r2_score(y, oof_ft):.4f}")
print(f"fine-tuning: {diff.mean():+.4f} ± {diff.std(ddof=1) / np.sqrt(5):.4f} (SE), better in {(diff > 0).sum()}/5 folds")
oof_ft.to_csv("oof_tabpfn_ft.csv")

Keep fine-tuning only if it is better in at least 4 of 5 folds. If it is, run the cells below; otherwise the plain TabPFN stays.

In [ ]:
# Full model: fine-tune on all 1152 training rows and predict the 776 test rows (sample.csv format, for Stacking/stacking.ipynb)

ft = FinetunedTabPFNRegressor(device="cuda", random_state=0).fit(X, y)
pd.Series(ft.predict(X_test), index=X_test.index, name="y").to_csv("submission_tabpfn_ft.csv")

In [ ]:
from google.colab import files

files.download("oof_tabpfn_ft.csv")          # -> Stacking/
files.download("submission_tabpfn_ft.csv")   # -> TabPFN/